# Logistic Regression Baseline - CLINC150

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [3]:
MAX_FEATURES = 5000

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), min_df=2, max_df=0.95)

train_features = vectorizer.fit_transform(train_texts).toarray()
val_features = vectorizer.transform(val_texts).toarray()
test_features = vectorizer.transform(test_texts).toarray()

print(f"TF-IDF dim: {train_features.shape[1]}")

TF-IDF dim: 2375


In [4]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


BATCH_SIZE = 128

train_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TextDataset(test_features, test_labels), batch_size=BATCH_SIZE, shuffle=False)

In [5]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)


model = LogisticRegression(train_features.shape[1], num_classes).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} params")

358,776 params


In [6]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            total_loss += criterion(outputs, labels).item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

In [7]:
LEARNING_RATE = 0.01
NUM_EPOCHS = 50
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_f1, best_model_state, patience_counter, best_epoch = 0, None, 0, 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_preds, vl_true = evaluate(model, val_loader, criterion)
    vl_f1 = f1_score(vl_true, vl_preds, average='macro', zero_division=0)
    vl_acc = accuracy_score(vl_true, vl_preds)

    mark = ""
    if vl_f1 > best_val_f1:
        best_val_f1, best_model_state, best_epoch, patience_counter = vl_f1, model.state_dict().copy(), epoch + 1, 0
        mark = " *"
    else:
        patience_counter += 1

    print(f"{epoch+1:2d}/{NUM_EPOCHS}  tr_loss={tr_loss:.4f} tr_acc={tr_acc:.1f}%  vl_loss={vl_loss:.4f} vl_acc={vl_acc*100:.1f}% vl_f1={vl_f1:.4f}{mark}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping. Best epoch: {best_epoch}, F1: {best_val_f1:.4f}")
        break

training_time = time.time() - start_time
print(f"Time: {training_time:.1f}s")

 1/50  tr_loss=4.9334 tr_acc=34.2%  vl_loss=4.8387 vl_acc=47.9% vl_f1=0.4577 *
 2/50  tr_loss=4.5516 tr_acc=92.0%  vl_loss=4.6386 vl_acc=56.1% vl_f1=0.5539 *
 3/50  tr_loss=4.1852 tr_acc=95.8%  vl_loss=4.4435 vl_acc=59.2% vl_f1=0.5802 *
 4/50  tr_loss=3.8284 tr_acc=96.9%  vl_loss=4.2525 vl_acc=59.5% vl_f1=0.5819 *
 5/50  tr_loss=3.4775 tr_acc=97.2%  vl_loss=4.0690 vl_acc=60.6% vl_f1=0.5955 *
 6/50  tr_loss=3.1386 tr_acc=97.9%  vl_loss=3.8908 vl_acc=61.5% vl_f1=0.6005 *
 7/50  tr_loss=2.8121 tr_acc=98.0%  vl_loss=3.7198 vl_acc=60.8% vl_f1=0.5947
 8/50  tr_loss=2.5047 tr_acc=98.0%  vl_loss=3.5582 vl_acc=61.8% vl_f1=0.6064 *
 9/50  tr_loss=2.2174 tr_acc=98.0%  vl_loss=3.4050 vl_acc=61.6% vl_f1=0.6062
10/50  tr_loss=1.9552 tr_acc=98.2%  vl_loss=3.2640 vl_acc=61.6% vl_f1=0.6049
11/50  tr_loss=1.7198 tr_acc=98.2%  vl_loss=3.1333 vl_acc=61.9% vl_f1=0.6087 *
12/50  tr_loss=1.5108 tr_acc=98.2%  vl_loss=3.0144 vl_acc=62.3% vl_f1=0.6140 *
13/50  tr_loss=1.3303 tr_acc=98.2%  vl_loss=2.9083 vl_acc=

In [10]:
model.load_state_dict(best_model_state)
_, test_preds, test_true = evaluate(model, test_loader, criterion)

acc = accuracy_score(test_true, test_preds)
p = precision_score(test_true, test_preds, average='macro', zero_division=0)
r = recall_score(test_true, test_preds, average='macro', zero_division=0)
f1_m = f1_score(test_true, test_preds, average='macro', zero_division=0)
f1_w = f1_score(test_true, test_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

Accuracy:   0.5173
Precision:  0.5434
Recall:     0.5777
F1 macro:   0.5336
F1 weighted:0.4967


In [11]:
results = {
    "model_type": "LogisticRegression",
    "optimization": "baseline",
    "best_epoch": best_epoch,
    "training_time_seconds": round(training_time, 2),
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "hyperparameters": {
        "max_features": MAX_FEATURES,
        "ngram_range": [1, 2],
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": NUM_EPOCHS,
        "early_stopping_patience": PATIENCE
    }
}

with open('results/results_lr_baseline.json', 'w') as f:
    json.dump(results, f, indent=4)

print("Saved: results/results_lr_baseline.json")

Saved: results/results_lr_baseline.json
